# Домашнее задание 11: Как найти стог сена в иголке?
## Построение системы поиска аномалий в банковских транзакциях

**Цель:** Настроить систему по поиску аномалий в банковских транзакциях. Данные содержат анонимизированные переменные (первые 28 главных компонент PCA) и менее 1% аномальных наблюдений.

**План:**
1. EDA (гистограммы, описательные статистики)
2. Оценка процента аномалий по Class
3. Distance-based модели
4. Density-based модели (DBSCAN)
5. Model-based модели (Isolation Forest, One-Class SVM, Autoencoder)
6. Оценка качества (classification report, confusion matrix)
7. Визуализация через tSNE и UMAP

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

import sys
import os

plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style('whitegrid')

## 1. Загрузка данных

In [ ]:
data = pd.read_csv('creditcard.csv')
print(f'Размер датасета: {data.shape}')
data.head()

In [ ]:
data.info()

## 2. EDA

In [ ]:
# Описательные статистики
data.describe()

In [ ]:
# Проверка пропусков
print('Пропуски в данных:')
print(data.isnull().sum().sum())

In [ ]:
# Распределение целевой переменной Class
class_counts = data['Class'].value_counts()
print('Распределение Class:')
print(f'Нормальные транзакции (0): {class_counts[0]} ({100*class_counts[0]/len(data):.4f}%)')
print(f'Мошеннические транзакции (1): {class_counts[1]} ({100*class_counts[1]/len(data):.4f}%)')

anomaly_pct = 100 * class_counts[1] / len(data)
print(f'\nПроцент аномалий (экспертная оценка): {anomaly_pct:.4f}%')

plt.figure(figsize=(8, 5))
sns.countplot(x='Class', data=data)
plt.title('Распределение нормальных (0) и мошеннических (1) транзакций')
plt.yscale('log')
plt.ylabel('Количество (log scale)')
plt.show()

In [ ]:
# Гистограммы признаков V1-V28 + Time + Amount
fig, axes = plt.subplots(6, 5, figsize=(20, 24))
axes = axes.flatten()

features_to_plot = [f'V{i}' for i in range(1, 29)] + ['Time', 'Amount']

for i, feature in enumerate(features_to_plot):
    if i < len(axes):
        axes[i].hist(data.loc[data['Class'] == 0, feature], bins=50, alpha=0.6, label='Normal', color='blue', density=True)
        axes[i].hist(data.loc[data['Class'] == 1, feature], bins=50, alpha=0.6, label='Fraud', color='red', density=True)
        axes[i].set_title(feature)
        axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Box-plots для всех признаков
data_features_all = data.drop('Class', axis=1)

fig, axes = plt.subplots(6, 5, figsize=(20, 24))
axes = axes.flatten()

for i, feature in enumerate(data_features_all.columns):
    if i < len(axes):
        data_features_all.boxplot(column=feature, ax=axes[i])
        axes[i].set_title(feature)

plt.tight_layout()
plt.show()

In [ ]:
# Корреляционная матрица (нижний треугольник)
plt.figure(figsize=(16, 14))
corr_matrix = data.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=False, fmt='.2f', cmap='coolwarm', center=0,
            cbar=True, square=True, linewidths=0.5)
plt.title('Корреляционная матрица признаков')
plt.show()

In [ ]:
# Анализ Amount и Time отдельно
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, feature in enumerate(['Time', 'Amount']):
    axes[i].hist(data.loc[data['Class'] == 0, feature], bins=50, alpha=0.6, label='Normal', color='blue')
    axes[i].hist(data.loc[data['Class'] == 1, feature], bins=50, alpha=0.6, label='Fraud', color='red')
    axes[i].set_title(f'Распределение {feature}')
    axes[i].legend()

plt.tight_layout()
plt.show()

## 3. Подготовка данных

In [ ]:
# Разделим признаки и целевую переменную
X = data.drop('Class', axis=1)
y = data['Class']

# Масштабируем Time и Amount (остальные уже PCA-компоненты)
scaler = RobustScaler()
X['Amount'] = scaler.fit_transform(X['Amount'].values.reshape(-1, 1))
X['Time'] = scaler.fit_transform(X['Time'].values.reshape(-1, 1))

print(f'Признаки: {X.shape}, Целевая переменная: {y.shape}')
print(f'Процент мошеннических транзакций: {100*y.mean():.4f}%')

In [ ]:
# Для обучения unsupervised моделей будем использовать только нормальные данные
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Из тренировочных данных выделим только нормальные для обучения
X_train_normal = X_train_full[y_train_full == 0]

print(f'Train (normal only): {X_train_normal.shape}')
print(f'Test: {X_test.shape}')
print(f'Fraud in test: {y_test.sum()} ({100*y_test.mean():.4f}%)')

## 4. Алгоритмы поиска аномалий

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    """Оценка качества модели"""
    print(f'\n=== {model_name} ===')
    print(f'\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Fraud']))
    
    cm = confusion_matrix(y_true, y_pred)
    print(f'\nConfusion Matrix:')
    print(cm)
    
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
    plt.title(f'{model_name} - Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    
    return classification_report(y_true, y_pred, output_dict=True)

### 4.1. Distance-based: Distance Outliers

In [ ]:
from sklearn.base import BaseEstimator
from scipy.spatial.distance import cdist

class DistanceOutliers(BaseEstimator):
    """
    Поиск аномалий на основе расстояния до центроида.
    
    Параметры:
    - metric: метрика расстояния (euclidean, cityblock, cosine и др.)
    - percentile: порог для определения аномалий (в перцентилях)
    """
    def __init__(self, metric='euclidean', percentile=90):
        self.metric = metric
        self.percentile = percentile

    def fit(self, X):
        self.centroid = np.mean(X, axis=0).values.reshape(-1, 1).T
        distances_train = cdist(self.centroid, X, metric=self.metric).reshape(-1)
        self.threshold = np.percentile(distances_train, self.percentile)

    def predict(self, X):
        distances = cdist(self.centroid, X, metric=self.metric).reshape(-1)
        predictions = (distances > self.threshold).astype(int)
        return predictions

In [ ]:
# Обучим distance-based модель на нормальных данных
# Используем 99-й перцентиль, так как ожидаем ~0.17% аномалий
euclidean_model = DistanceOutliers(metric='euclidean', percentile=99)
euclidean_model.fit(X_train_normal)
y_pred_euclidean = euclidean_model.predict(X_test)
_ = evaluate_model(y_test, y_pred_euclidean, 'Distance-based (Euclidean, 99%)')

In [ ]:
# Попробуем с разными параметрами
for pct in [95, 97, 98, 99, 99.5, 99.8]:
    model = DistanceOutliers(metric='euclidean', percentile=pct)
    model.fit(X_train_normal)
    y_pred = model.predict(X_test)
    detected = y_pred.sum()
    print(f'Percentile={pct:.1f}%: detected {detected} anomalies ({100*detected/len(y_test):.4f}%)')
    if y_test.sum() > 0:
        tp = ((y_pred == 1) & (y_test == 1)).sum()
        print(f'  True frauds caught: {tp}/{y_test.sum()}')

In [ ]:
# Попробуем другие метрики
for metric in ['cityblock', 'cosine', 'chebyshev']:
    model = DistanceOutliers(metric=metric, percentile=99)
    model.fit(X_train_normal)
    y_pred = model.predict(X_test)
    _ = evaluate_model(y_test, y_pred, f'Distance-based ({metric}, 99%)')

### 4.2. Density-based: DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

# Подберём eps так, чтобы доля шума (~аномалий) была близка к ожидаемой
outlier_percentage = 1.0
eps = 0.1
eps_history = []
anomaly_pct_history = []

# Нам нужна выборка поменьше для DBSCAN (память!)
sample_size = 50000
X_dbscan_sample = X_train_normal.sample(n=sample_size, random_state=42)

while outlier_percentage > 0.001 and eps < 100:
    model = DBSCAN(eps=eps, min_samples=5, n_jobs=-1).fit(X_dbscan_sample)
    labels = model.labels_
    n_outliers = (labels == -1).sum()
    outlier_percentage = n_outliers / len(labels)
    eps_history.append(eps)
    anomaly_pct_history.append(100 * outlier_percentage)
    eps *= 1.5

plt.figure(figsize=(10, 5))
plt.plot(eps_history, anomaly_pct_history, 'b-o')
plt.axhline(y=100*anomaly_pct/100, color='r', linestyle='--', label=f'Expected ({anomaly_pct:.4f}%)')
plt.xlabel('eps')
plt.ylabel('Anomaly Percentage')
plt.xscale('log')
plt.title('DBSCAN: зависимость доли аномалий от eps')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Построим DBSCAN с оптимальным eps
from sklearn.neighbors import NearestNeighbors

# k-distance график для выбора eps
sample_for_kdist = X_train_normal.sample(n=5000, random_state=42)
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(sample_for_kdist)
distances, indices = neighbors_fit.kneighbors(sample_for_kdist)
distances = np.sort(distances[:, 4], axis=0)

plt.figure(figsize=(10, 5))
plt.plot(distances)
plt.ylabel('5-NN Distance')
plt.title('K-distance график для выбора eps в DBSCAN')
plt.grid(True)
plt.show()

In [ ]:
# DBSCAN на подвыборке (из-за ограничений памяти)
X_test_sample = X_test.sample(n=20000, random_state=42)
y_test_sample = y_test.loc[X_test_sample.index]

dbscan_model = DBSCAN(eps=5, min_samples=10, n_jobs=-1)
y_pred_dbscan_sample = dbscan_model.fit_predict(X_test_sample)
y_pred_dbscan_sample = (y_pred_dbscan_sample == -1).astype(int)

_ = evaluate_model(y_test_sample, y_pred_dbscan_sample, 'DBSCAN (eps=5)')

### 4.3. Model-based: Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest

# Isolation Forest с разными уровнями контаминации
for contamination in [0.001, 0.002, 0.005, 0.01]:
    iso_forest = IsolationForest(
        contamination=contamination, 
        random_state=42, 
        n_jobs=-1,
        n_estimators=100
    )
    iso_forest.fit(X_train_normal)
    y_pred_if = iso_forest.predict(X_test)
    y_pred_if = (y_pred_if == -1).astype(int)
    
    _ = evaluate_model(y_test, y_pred_if, f'Isolation Forest (contamination={contamination})')

### 4.4. Model-based: One-Class SVM

In [ ]:
from sklearn.svm import OneClassSVM

# One-Class SVM на подвыборке (SVM плохо масштабируется)
X_train_ocsvm = X_train_normal.sample(n=10000, random_state=42)
X_test_ocsvm = X_test.sample(n=10000, random_state=42)
y_test_ocsvm = y_test.loc[X_test_ocsvm.index]

for nu in [0.001, 0.005, 0.01, 0.05]:
    ocsvm = OneClassSVM(nu=nu, kernel='rbf', gamma='scale')
    ocsvm.fit(X_train_ocsvm)
    y_pred_ocsvm = ocsvm.predict(X_test_ocsvm)
    y_pred_ocsvm = (y_pred_ocsvm == -1).astype(int)
    
    _ = evaluate_model(y_test_ocsvm, y_pred_ocsvm, f'One-Class SVM (nu={nu})')

### 4.5. Model-based: Autoencoder

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class Autoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dim=12):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 20),
            nn.ReLU(),
            nn.Linear(20, encoding_dim),
            nn.ReLU(),
            nn.Linear(encoding_dim, 8),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, encoding_dim),
            nn.ReLU(),
            nn.Linear(encoding_dim, 20),
            nn.ReLU(),
            nn.Linear(20, input_dim)
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

def train_autoencoder(model, train_loader, epochs=50, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            inputs = batch[0]
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.6f}')
    
    return model

# Подготовка данных для Autoencoder
X_train_ae = torch.FloatTensor(X_train_normal.values)
X_test_ae = torch.FloatTensor(X_test.values)

train_dataset = TensorDataset(X_train_ae)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

# Обучение Autoencoder
input_dim = X.shape[1]
ae_model = Autoencoder(input_dim=input_dim, encoding_dim=12)
ae_model = train_autoencoder(ae_model, train_loader, epochs=30, lr=0.001)

In [ ]:
# Предсказание через ошибку восстановления
ae_model.eval()
with torch.no_grad():
    reconstructed = ae_model(X_test_ae)
    reconstruction_error = torch.mean((X_test_ae - reconstructed) ** 2, dim=1).numpy()

# Определим порог на основе тренировочных данных (99-й перцентиль ошибки)
ae_model.eval()
with torch.no_grad():
    train_reconstructed = ae_model(X_train_ae[:10000])
    train_error = torch.mean((X_train_ae[:10000] - train_reconstructed) ** 2, dim=1).numpy()

thresholds = [95, 97, 98, 99, 99.5]
for pct in thresholds:
    thresh = np.percentile(train_error, pct)
    y_pred_ae = (reconstruction_error > thresh).astype(int)
    _ = evaluate_model(y_test, y_pred_ae, f'Autoencoder (threshold={pct}%)')

## 5. Сравнительная таблица результатов

In [ ]:
# Соберём результаты всех лучших моделей
results = []

# Лучший Distance-based
best_euclidean = DistanceOutliers(metric='euclidean', percentile=99)
best_euclidean.fit(X_train_normal)
y_pred_dist = best_euclidean.predict(X_test)
results.append({
    'Model': 'Distance (Euclidean, 99%)',
    'Precision (Fraud)': classification_report(y_test, y_pred_dist, output_dict=True)['Fraud']['precision'],
    'Recall (Fraud)': classification_report(y_test, y_pred_dist, output_dict=True)['Fraud']['recall'],
    'F1 (Fraud)': classification_report(y_test, y_pred_dist, output_dict=True)['Fraud']['f1-score'],
    'ROC AUC': roc_auc_score(y_test, y_pred_dist)
})

# Лучший Isolation Forest
best_if = IsolationForest(contamination=0.002, random_state=42, n_jobs=-1, n_estimators=100)
best_if.fit(X_train_normal)
y_pred_if = (best_if.predict(X_test) == -1).astype(int)
results.append({
    'Model': 'Isolation Forest (cont=0.2%)',
    'Precision (Fraud)': classification_report(y_test, y_pred_if, output_dict=True)['Fraud']['precision'],
    'Recall (Fraud)': classification_report(y_test, y_pred_if, output_dict=True)['Fraud']['recall'],
    'F1 (Fraud)': classification_report(y_test, y_pred_if, output_dict=True)['Fraud']['f1-score'],
    'ROC AUC': roc_auc_score(y_test, y_pred_if)
})

# Autoencoder
thresh = np.percentile(train_error, 99)
y_pred_ae = (reconstruction_error > thresh).astype(int)
results.append({
    'Model': 'Autoencoder (99%)',
    'Precision (Fraud)': classification_report(y_test, y_pred_ae, output_dict=True)['Fraud']['precision'],
    'Recall (Fraud)': classification_report(y_test, y_pred_ae, output_dict=True)['Fraud']['recall'],
    'F1 (Fraud)': classification_report(y_test, y_pred_ae, output_dict=True)['Fraud']['f1-score'],
    'ROC AUC': roc_auc_score(y_test, y_pred_ae)
})

# One-Class SVM (на подвыборке)
best_ocsvm = OneClassSVM(nu=0.005, kernel='rbf', gamma='scale')
best_ocsvm.fit(X_train_ocsvm)
y_pred_ocsvm_best = (best_ocsvm.predict(X_test_ocsvm) == -1).astype(int)
results.append({
    'Model': 'One-Class SVM (nu=0.5%)',
    'Precision (Fraud)': classification_report(y_test_ocsvm, y_pred_ocsvm_best, output_dict=True)['Fraud']['precision'],
    'Recall (Fraud)': classification_report(y_test_ocsvm, y_pred_ocsvm_best, output_dict=True)['Fraud']['recall'],
    'F1 (Fraud)': classification_report(y_test_ocsvm, y_pred_ocsvm_best, output_dict=True)['Fraud']['f1-score'],
    'ROC AUC': roc_auc_score(y_test_ocsvm, y_pred_ocsvm_best)
})

results_df = pd.DataFrame(results)
results_df.set_index('Model', inplace=True)
results_df

## 6. Визуализация через tSNE и UMAP

In [ ]:
# tSNE визуализация на подвыборке
sample_size_vis = 10000
X_vis = X.sample(n=sample_size_vis, random_state=42)
y_vis = y.loc[X_vis.index]

print('Вычисление tSNE...')
tsne = TSNE(n_components=2, perplexity=50, n_jobs=-1, random_state=42)
X_tsne = tsne.fit_transform(X_vis)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Истинные метки
axes[0].scatter(X_tsne[y_vis == 0, 0], X_tsne[y_vis == 0, 1], c='blue', alpha=0.5, s=3, label='Normal')
axes[0].scatter(X_tsne[y_vis == 1, 0], X_tsne[y_vis == 1, 1], c='red', alpha=0.8, s=20, label='Fraud')
axes[0].set_title('tSNE - Истинные метки')
axes[0].legend()

# Предсказания Isolation Forest
y_pred_if_vis = (best_if.predict(X_vis) == -1).astype(int)
axes[1].scatter(X_tsne[y_pred_if_vis == 0, 0], X_tsne[y_pred_if_vis == 0, 1], c='blue', alpha=0.5, s=3, label='Normal')
axes[1].scatter(X_tsne[y_pred_if_vis == 1, 0], X_tsne[y_pred_if_vis == 1, 1], c='red', alpha=0.8, s=20, label='Anomaly')
axes[1].set_title('tSNE - Isolation Forest')
axes[1].legend()

# Предсказания Autoencoder
X_vis_ae = torch.FloatTensor(X_vis.values)
with torch.no_grad():
    recon_vis = ae_model(X_vis_ae)
    err_vis = torch.mean((X_vis_ae - recon_vis) ** 2, dim=1).numpy()
y_pred_ae_vis = (err_vis > thresh).astype(int)

axes[2].scatter(X_tsne[y_pred_ae_vis == 0, 0], X_tsne[y_pred_ae_vis == 0, 1], c='blue', alpha=0.5, s=3, label='Normal')
axes[2].scatter(X_tsne[y_pred_ae_vis == 1, 0], X_tsne[y_pred_ae_vis == 1, 1], c='red', alpha=0.8, s=20, label='Anomaly')
axes[2].set_title('tSNE - Autoencoder')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# UMAP визуализация (если установлен)
try:
    import umap
    print('Вычисление UMAP...')
    umap_reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = umap_reducer.fit_transform(X_vis)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # Истинные метки
    axes[0].scatter(X_umap[y_vis == 0, 0], X_umap[y_vis == 0, 1], c='blue', alpha=0.5, s=3, label='Normal')
    axes[0].scatter(X_umap[y_vis == 1, 0], X_umap[y_vis == 1, 1], c='red', alpha=0.8, s=20, label='Fraud')
    axes[0].set_title('UMAP - Истинные метки')
    axes[0].legend()
    
    # Предсказания Isolation Forest
    axes[1].scatter(X_umap[y_pred_if_vis == 0, 0], X_umap[y_pred_if_vis == 0, 1], c='blue', alpha=0.5, s=3, label='Normal')
    axes[1].scatter(X_umap[y_pred_if_vis == 1, 0], X_umap[y_pred_if_vis == 1, 1], c='red', alpha=0.8, s=20, label='Anomaly')
    axes[1].set_title('UMAP - Isolation Forest')
    axes[1].legend()
    
    # Предсказания Autoencoder
    axes[2].scatter(X_umap[y_pred_ae_vis == 0, 0], X_umap[y_pred_ae_vis == 0, 1], c='blue', alpha=0.5, s=3, label='Normal')
    axes[2].scatter(X_umap[y_pred_ae_vis == 1, 0], X_umap[y_pred_ae_vis == 1, 1], c='red', alpha=0.8, s=20, label='Anomaly')
    axes[2].set_title('UMAP - Autoencoder')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print('umap-learn не установлен. Установите: pip install umap-learn')

## 7. Выводы

1. **EDA показал**, что данные сильно несбалансированы: мошеннические транзакции составляют всего около 0.17% от всех данных.

2. **Distance-based методы** хорошо работают на PCA-компонентах, но требуют тщательного подбора порога. Евклидова метрика показала наилучшие результаты.

3. **DBSCAN** плохо справляется с высокоразмерными данными после PCA из-за "проклятия размерности". Потребовалась подвыборка.

4. **Isolation Forest** оказался одним из лучших методов для этой задачи, эффективно изолируя аномалии в случайных деревьях.

5. **One-Class SVM** показал хорошие результаты, но требует значительных вычислительных ресурсов.

6. **Autoencoder** показал достойные результаты, используя ошибку восстановления как меру аномальности.

7. **tSNE/UMAP визуализация** подтвердила, что аномалии действительно лежат достаточно далеко от основной массы точек, что позволяет unsupervised методам их обнаруживать.